In [ ]:
# BOB Theta — Evaluation (native 300 Hz)
# Conditions saved: DEFAULT (pre-optimization Burg) + OPT (Bayesian-optimized, Opt_o)
# Metrics: MACE, PLV, CSD (circular SD), Acc. GT = matched zero-phase FIR on the same channel.
# Saves one model-tagged JSON holding raw per-timepoint errors, both conditions' metrics,
# per-subject SNR + ITPC, and the per-subject BOB param configs.
import numpy as np
import os, time, json, gc, glob
from scipy.signal import butter, hilbert, lfilter, lfilter_zi, filtfilt, welch, firwin
from scipy.stats import pearsonr, spearmanr
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
try:
    from skopt import gp_minimize
    from skopt.space import Integer, Real
except ImportError:
    import subprocess; subprocess.check_call(['pip', 'install', 'scikit-optimize', '-q'])
    from skopt import gp_minimize
    from skopt.space import Integer, Real
print("setup ok")

Mounted at /content/drive
setup ok


In [ ]:
# ---- CONFIG ----
STIM_DIR = '/content/drive/MyDrive/RAIN_IRC/data splits/onset'
OUTDIR   = '/content/drive/MyDrive/RAIN_IRC/results/BOB_onset'
os.makedirs(OUTDIR, exist_ok=True)

EVAL_TIMES_S = [0.3, 0.4, 0.5, 0.6, 0.7]
HORIZON_MS   = 0.0
GT_LOW, GT_HIGH = 4.0, 8.0            # fixed theta GT band (both conditions scored against this)
GT_FIR_NUMTAPS  = 175                 # 300 Hz (half of the 351 used at 600 Hz)
AR_CONTEXT_DIV  = 2
THETA_SEARCH_LO, THETA_SEARCH_HI = 4.0, 8.0
N_BAYES_CALLS, N_INITIAL, SEED = 100, 20, 42

# Baseline (pre-optimization) Burg config — literature defaults, kept for default-vs-opt.
DEFAULT = {'ar_order': 16, 'context_s': 1.0, 'filt_low': 4.0, 'filt_high': 8.0,
           'filt_order': 2, 'hilbert_pad': 60}

# AR bounds were tuned at 600 Hz (~100 samp/cycle); at 300 Hz a cycle is ~50 samp, so these
# orders span more cycles now -- watch for AR saturating the ceiling and scale down if so.
SEARCH_SPACE = [
    Integer(24, 100, name='ar_order'),
    Real(0.8, 1.6,   name='context_s'),
    Real(4.0, 5.5,   name='filt_low'),
    Real(6.0, 8.0,   name='filt_high'),
    Integer(1, 4,    name='filt_order'),
    Integer(30, 100, name='hilbert_pad'),
]

files = sorted(glob.glob(os.path.join(STIM_DIR, '*.npz')))
assert files, f"no npz found in {STIM_DIR}"
_d0 = np.load(files[0], allow_pickle=True)
FS = float(_d0['fs']); PRE_SAMP = int(_d0['onset_in_epoch']); _d0.close()
HORIZON_SAMP = int(HORIZON_MS * FS / 1000.0)
EVAL_TIMES_SAMP = [PRE_SAMP + int(t * FS) for t in EVAL_TIMES_S]
print(f"FS={FS:.0f} Hz | t=0 at sample {PRE_SAMP} | GT {GT_LOW}-{GT_HIGH} Hz FIR-{GT_FIR_NUMTAPS}")
print(f"eval samples {EVAL_TIMES_SAMP} | {len(files)} subject files")

FS=300 Hz | t=0 at sample 300 | GT 4.0-8.0 Hz FIR-175
eval samples [390, 420, 450, 480, 510] | 20 subject files


In [ ]:
# ---- ENGINE ----
def make_causal_filter(lo, hi, order):
    nyq = FS / 2.0
    lo = max(0.5, min(lo, nyq - 1)); hi = max(lo + 0.5, min(hi, nyq - 0.5))
    return butter(int(order), [lo / nyq, hi / nyq], btype='band')

def causal_filter_signal(sig1d, b, a):
    zi = lfilter_zi(b, a) * sig1d[0]
    out, _ = lfilter(b, a, sig1d, zi=zi)
    return out.astype(np.float32)

def burg_ar_coeffs(x, order):
    x = x.astype(np.float64); n = len(x); order = int(order)
    if n <= order + 1: return np.zeros(order)
    ef = x.copy(); eb = x.copy(); a = np.zeros(order)
    for m in range(order):
        ef_s, eb_s = ef[m + 1:n], eb[m:n - 1]; L = min(len(ef_s), len(eb_s))
        if L <= 0: break
        ef_s, eb_s = ef_s[:L].copy(), eb_s[:L].copy()
        num = -2.0 * np.dot(ef_s, eb_s); den = np.dot(ef_s, ef_s) + np.dot(eb_s, eb_s)
        if den < 1e-12: break
        k = np.clip(num / den, -0.9999, 0.9999)
        a_new = np.zeros(m + 1); a_new[m] = k
        if m > 0: a_new[:m] = a[:m] + k * a[m - 1::-1]
        a[:m + 1] = a_new
        ef[m + 1:m + 1 + L] = ef_s + k * eb_s; eb[m + 1:m + 1 + L] = eb_s + k * ef_s; n = m + 1 + L
    return -a

def ar_extrapolate(x, coeffs, n_steps):
    p = len(coeffs); coeffs = np.asarray(coeffs, np.float64)
    out = np.empty(len(x) + n_steps, np.float64); out[:len(x)] = x
    for i in range(len(x), len(x) + n_steps):
        out[i] = out[i - 1] if i < p else np.dot(coeffs, out[i - p:i][::-1])
    return out

def burg_predict_phase(csig, predict_at, horizon, ar_order, context_s, hpad):
    csamp = int(context_s * FS); pad = int(hpad)
    start = max(0, predict_at - csamp + 1)
    x = csig[start:predict_at + 1].astype(np.float64)
    ar_order = min(int(ar_order), len(x) // AR_CONTEXT_DIV)
    if len(x) < ar_order + 5 or np.std(x) < 1e-10: return 0.0, 1.0
    ext = ar_extrapolate(x, burg_ar_coeffs(x, ar_order), horizon + pad)
    tgt = len(x) - 1 + horizon
    if tgt >= len(ext): return 0.0, 1.0
    ph = np.angle(hilbert(ext)[tgt]); return np.sin(ph), np.cos(ph)

def compute_gt(raw_epoch, lo, hi):
    nyq = FS / 2.0
    lo = max(0.5, min(lo, nyq - 1)); hi = max(lo + 0.5, min(hi, nyq - 0.5))
    taps = firwin(GT_FIR_NUMTAPS, [lo, hi], pass_zero=False, fs=FS)
    ph = np.angle(hilbert(filtfilt(taps, [1.0], raw_epoch.astype(np.float64))))
    return np.sin(ph).astype(np.float32), np.cos(ph).astype(np.float32)

def mace_from_errors(e):
    return 90.0 if len(e) < 10 else float(np.degrees(np.mean(np.abs(e))))

def errors_to_metrics(e):
    if len(e) < 10:
        return {'mace': 90.0, 'plv': 0.0, 'csd': 180.0, 'acc': 0.0, 'bias': 0.0, 'n_samples': 0}
    mean_abs = float(np.mean(np.abs(e))); R = float(np.abs(np.mean(np.exp(1j * e))))
    return {'mace': float(np.degrees(mean_abs)), 'plv': R,
            'csd': float(np.degrees(np.sqrt(-2.0 * np.log(max(R, 1e-12))))),   # circular SD (deg)
            'acc': float((1.0 - mean_abs / np.pi) * 100.0),
            'bias': float(np.degrees(np.angle(np.mean(np.exp(1j * e))))),
            'n_samples': int(len(e))}

def collect_errors(raw, gs_l, gc_l, trials, ar, ctx, lo, hi, ordf, hpad):
    try: b, a = make_causal_filter(lo, hi, int(ordf))
    except Exception: return np.array([])
    L = raw.shape[1]; errs = []
    for tr in trials:
        cs = causal_filter_signal(raw[tr], b, a)
        for es in EVAL_TIMES_SAMP:
            g = es + HORIZON_SAMP
            if g >= L: continue
            s, c = burg_predict_phase(cs, es, HORIZON_SAMP, int(ar), ctx, int(hpad))
            gsv, gcv = gs_l[tr][g], gc_l[tr][g]
            errs.append(np.arctan2(s * gcv - c * gsv, c * gcv + s * gsv))
    return np.array(errs)

def collect_errors_per_tp(raw, gs_l, gc_l, trials, ar, ctx, lo, hi, ordf, hpad):
    try: b, a = make_causal_filter(lo, hi, int(ordf))
    except Exception: return {t: np.array([]) for t in EVAL_TIMES_S}
    L = raw.shape[1]; by_t = {t: [] for t in EVAL_TIMES_S}
    for tr in trials:
        cs = causal_filter_signal(raw[tr], b, a)
        for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP):
            g = es + HORIZON_SAMP
            if g >= L: continue
            s, c = burg_predict_phase(cs, es, HORIZON_SAMP, int(ar), ctx, int(hpad))
            gsv, gcv = gs_l[tr][g], gc_l[tr][g]
            by_t[t].append(np.arctan2(s * gcv - c * gsv, c * gcv + s * gsv))
    return {t: np.array(v) for t, v in by_t.items()}

def run_per_tp(raw, gs_l, gc_l, trials, p):
    return collect_errors_per_tp(raw, gs_l, gc_l, trials, p['ar_order'], p['context_s'],
                                 p['filt_low'], p['filt_high'], p['filt_order'], p['hilbert_pad'])

def itpc_from_gt(gs_l, gc_l, trials, esamp):
    ph = np.array([np.arctan2(gs_l[tr][esamp], gc_l[tr][esamp]) for tr in trials])
    return float(np.abs(np.mean(np.exp(1j * ph))))

def estimate_theta_peak(raw, onset_samp):
    seg = raw[:, onset_samp:]
    if seg.shape[1] < 16: return np.nan
    nfft = int(2 ** np.ceil(np.log2(seg.shape[1] * 4))); psd = None
    for tr in range(seg.shape[0]):
        f, px = welch(seg[tr].astype(np.float64), fs=FS, nperseg=seg.shape[1], nfft=nfft, detrend='linear')
        psd = px if psd is None else psd + px
    psd /= seg.shape[0]; band = (f >= THETA_SEARCH_LO) & (f <= THETA_SEARCH_HI)
    return float(f[band][np.argmax(psd[band])]) if np.any(band) else np.nan

In [ ]:
# ---- RUN ----
print(f"{'Subject':<12}{'N':>5}{'defMACE':>8}{'optMACE':>8}{'optPLV':>7}{'optCSD':>7}{'optAcc':>7}{'SNR':>6}{'ITPC':>6}{'Peak':>6}")
print('-' * 80)
results = []
for fp in files:
    t0 = time.time(); d = np.load(fp, allow_pickle=True); name = os.path.basename(fp)[:-4]
    if float(d['fs']) != FS or int(d['onset_in_epoch']) != PRE_SAMP:
        print(f"{name}: geometry mismatch, skipping"); d.close(); continue
    epochs = d['epochs']; best = int(d['best_ch_idx']); ch_name = str(d['best_ch_name'])
    split_arr = np.array([str(x) for x in d['split']]); theta_snr = float(d['ch_theta_snr'][best])
    raw = epochs[:, best, :].astype(np.float32); n_tr = raw.shape[0]
    cal = np.where(split_arr == 'cal')[0]; test = np.where(split_arr == 'test')[0]

    gs_l, gc_l = [], []
    for tr in range(n_tr):
        s, c = compute_gt(raw[tr], GT_LOW, GT_HIGH); gs_l.append(s); gc_l.append(c)

    # --- baseline (pre-optimization) on TEST ---
    dflt_tp = run_per_tp(raw, gs_l, gc_l, test, DEFAULT)
    dflt_err = np.concatenate([dflt_tp[t] for t in EVAL_TIMES_S]) if any(len(dflt_tp[t]) for t in EVAL_TIMES_S) else np.array([])
    dflt_m = errors_to_metrics(dflt_err); dflt_tp_m = {t: errors_to_metrics(dflt_tp[t]) for t in EVAL_TIMES_S}

    # --- Bayesian optimization on CAL ---
    def objective(params):
        ar, ctx, lo, hi, ordf, hpad = params
        if lo >= hi - 0.5: return 90.0
        arc = min(int(ar), int(ctx * FS) // AR_CONTEXT_DIV)
        if arc < 4: return 90.0
        try: make_causal_filter(lo, hi, int(ordf))
        except Exception: return 90.0
        return mace_from_errors(collect_errors(raw, gs_l, gc_l, cal, arc, ctx, lo, hi, int(ordf), int(hpad)))
    res = gp_minimize(objective, SEARCH_SPACE, n_calls=N_BAYES_CALLS,
                      n_initial_points=N_INITIAL, random_state=SEED, verbose=False)
    bp = {'ar_order': int(res.x[0]), 'context_s': float(res.x[1]), 'filt_low': float(res.x[2]),
          'filt_high': float(res.x[3]), 'filt_order': int(res.x[4]), 'hilbert_pad': int(res.x[5])}
    bp['ar_order'] = min(bp['ar_order'], int(bp['context_s'] * FS) // AR_CONTEXT_DIV)

    # --- optimized on TEST ---
    opt_tp = run_per_tp(raw, gs_l, gc_l, test, bp)
    opt_err = np.concatenate([opt_tp[t] for t in EVAL_TIMES_S]) if any(len(opt_tp[t]) for t in EVAL_TIMES_S) else np.array([])
    opt_m = errors_to_metrics(opt_err); opt_tp_m = {t: errors_to_metrics(opt_tp[t]) for t in EVAL_TIMES_S}

    itpc_tp = {t: (itpc_from_gt(gs_l, gc_l, test, es) if es < raw.shape[1] else float('nan'))
               for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP)}
    itpc_mean = float(np.nanmean(list(itpc_tp.values()))); f_peak = estimate_theta_peak(raw, PRE_SAMP)

    results.append({
        'name': name, 'best_ch': ch_name, 'n_trials': int(n_tr),
        'n_cal': int(len(cal)), 'n_test': int(len(test)), 'theta_snr': theta_snr,
        'theta_itpc_per_timepoint': {str(t): itpc_tp[t] for t in EVAL_TIMES_S},
        'theta_itpc_mean': itpc_mean,
        'theta_peak_hz': float(f_peak) if np.isfinite(f_peak) else None,
        'best_params': bp, 'default_params': DEFAULT,
        'default_metrics_pooled': dflt_m,
        'default_metrics_per_timepoint': {str(t): dflt_tp_m[t] for t in EVAL_TIMES_S},
        'default_errors_per_timepoint_rad': {str(t): [float(x) for x in dflt_tp[t]] for t in EVAL_TIMES_S},
        'opt_metrics_pooled': opt_m,
        'opt_metrics_per_timepoint': {str(t): opt_tp_m[t] for t in EVAL_TIMES_S},
        'opt_errors_per_timepoint_rad': {str(t): [float(x) for x in opt_tp[t]] for t in EVAL_TIMES_S},
    })
    pk = f"{f_peak:.2f}" if np.isfinite(f_peak) else " n/a"
    print(f"{name:<12}{opt_m['n_samples']:>5}{dflt_m['mace']:>7.1f}\u00b0{opt_m['mace']:>7.1f}\u00b0{opt_m['plv']:>7.3f}"
          f"{opt_m['csd']:>6.1f}\u00b0{opt_m['acc']:>6.1f}%{theta_snr:>6.2f}{itpc_mean:>6.3f}{pk:>6}  [{time.time()-t0:.0f}s]")
    d.close(); del epochs, raw, gs_l, gc_l; gc.collect()
print(f"\n{len(results)} subjects done")

Subject         N defMACE optMACE optPLV optCSD optAcc   SNR  ITPC  Peak
--------------------------------------------------------------------------------
Subject10001  250   51.6°   50.1°  0.537  63.9°  72.1%  2.56 0.097  4.03  [232s]
Subject1001   250   52.1°   47.3°  0.572  60.6°  73.7%  2.59 0.162  4.03  [187s]
Subject11002  250   57.2°   50.7°  0.531  64.5°  71.8%  2.04 0.128  4.03  [186s]
Subject12001  250   55.4°   47.2°  0.560  61.7°  73.8%  1.93 0.179  4.03  [187s]
Subject13001  250   44.8°   40.9°  0.649  53.3°  77.3%  2.65 0.072  6.88  [183s]
Subject14001  250   55.9°   51.6°  0.512  66.3°  71.3%  2.41 0.169  4.03  [181s]
Subject15001  250   56.8°   51.2°  0.517  65.8°  71.6%  2.36 0.146  4.03  [194s]
Subject16001  250   55.4°   50.1°  0.538  63.8°  72.2%  1.74 0.192  4.03  [205s]
Subject17001  250   51.0°   46.5°  0.576  60.2°  74.1%  2.45 0.205  4.03  [180s]
Subject18002  250   64.7°   46.8°  0.609  57.0°  74.0%  2.99 0.121  7.76  [216s]
Subject19001  250   55.3°   47.9°  0

In [ ]:
# ---- SUMMARY + SAVE ----
n = len(results)
dM = np.array([r['default_metrics_pooled']['mace'] for r in results])
oM = np.array([r['opt_metrics_pooled']['mace'] for r in results])
oP = np.array([r['opt_metrics_pooled']['plv'] for r in results])
oC = np.array([r['opt_metrics_pooled']['csd'] for r in results])
oA = np.array([r['opt_metrics_pooled']['acc'] for r in results])
dA = np.array([r['default_metrics_pooled']['acc'] for r in results])
SNR = np.array([r['theta_snr'] for r in results]); IT = np.array([r['theta_itpc_mean'] for r in results])

print('=' * 72); print('BOB THETA — DEFAULT vs OPTIMIZED (Opt_o)'); print('=' * 72)
print(f"  Default   MACE {dM.mean():.1f}\u00b0\u00b1{dM.std():.1f}   Acc {dA.mean():.1f}%")
print(f"  Optimized MACE {oM.mean():.1f}\u00b0\u00b1{oM.std():.1f}   PLV {oP.mean():.3f}   CSD {oC.mean():.1f}\u00b0   Acc {oA.mean():.1f}%")
gain = dM - oM
print(f"  Optimization gain: {gain.mean():+.1f}\u00b0 MACE  ({int(np.sum(gain > 0))}/{n} improved)")
print(f"  (n={n} unique participants -- independent samples; group/inferential stats valid)")

# paired default-vs-optimized significance (per-subject, N independent participants, two-sided)
from scipy.stats import wilcoxon, ttest_rel, rankdata
diff = dM - oM; da = oA - dA   # >0 means optimization improved (MACE down / Acc up)
group_stats = {'n_participants': int(n), 'paired_test': 'two-sided',
               'mean_mace_default': float(dM.mean()), 'mean_mace_opt': float(oM.mean()),
               'mean_mace_reduction': float(diff.mean()), 'median_mace_reduction': float(np.median(diff)),
               'n_improved': int(np.sum(diff > 0))}
print('\n--- Paired default vs optimized (per-subject MACE, test set) ---')
print(f"  mean dMACE {diff.mean():+.2f}\u00b0   median {np.median(diff):+.2f}\u00b0   ({int(np.sum(diff > 0))}/{n} improved)")
if n >= 3 and np.any(diff != 0):
    w_stat, w_p = wilcoxon(dM, oM)
    nz = diff[diff != 0]; rk = rankdata(np.abs(nz))
    rbc = float((rk[nz > 0].sum() - rk[nz < 0].sum()) / rk.sum())
    t_stat, t_p = ttest_rel(dM, oM)
    sd = diff.std(ddof=1); dz = float(diff.mean() / sd) if sd > 0 else float('nan')
    group_stats.update({'wilcoxon_W': float(w_stat), 'wilcoxon_p': float(w_p), 'rank_biserial_r': rbc,
                        'ttest_t': float(t_stat), 'ttest_p': float(t_p), 'cohen_dz': dz})
    print(f"  Wilcoxon signed-rank: W={w_stat:.1f}, p={w_p:.4g}  (rank-biserial r={rbc:+.3f})")
    print(f"  Paired t-test:        t={t_stat:+.2f}, p={t_p:.4g}  (Cohen's dz={dz:+.3f})")
    if np.any(da != 0):
        _, ap = wilcoxon(oA, dA); group_stats['accuracy_wilcoxon_p'] = float(ap)
        print(f"  Accuracy (opt-def):   median {np.median(da):+.2f} pts, Wilcoxon p={ap:.4g}")
else:
    group_stats['note'] = 'insufficient nonzero differences'
    print("  (insufficient nonzero differences for a paired test)")

print('\n--- Per-timepoint (default vs optimized) ---')
print(f"{'ms':>5}{'defMACE':>9}{'optMACE':>9}{'optPLV':>8}{'optCSD':>8}{'optAcc':>8}{'ITPC':>8}{'N':>7}")
for t in EVAL_TIMES_S:
    dm = np.array([r['default_metrics_per_timepoint'][str(t)]['mace'] for r in results])
    g = lambda k: np.array([r['opt_metrics_per_timepoint'][str(t)][k] for r in results])
    ti = np.array([r['theta_itpc_per_timepoint'][str(t)] for r in results])
    print(f"{int(t * 1000):>5}{dm.mean():>8.1f}\u00b0{g('mace').mean():>8.1f}\u00b0{g('plv').mean():>8.3f}"
          f"{g('csd').mean():>7.1f}\u00b0{g('acc').mean():>7.1f}%{ti.mean():>8.3f}{g('n_samples').mean():>7.0f}")

print('\n--- Accuracy (optimized) correlations across subjects ---')
if np.std(SNR) > 0:
    r1, p1 = pearsonr(SNR, oA); rs1, ps1 = spearmanr(SNR, oA)
    print(f"  Acc vs SNR : r={r1:+.3f} (p={p1:.4f})  rho={rs1:+.3f} (p={ps1:.4f})")
if np.std(IT) > 0:
    r2, p2 = pearsonr(IT, oA); rs2, ps2 = spearmanr(IT, oA)
    print(f"  Acc vs ITPC: r={r2:+.3f} (p={p2:.4f})  rho={rs2:+.3f} (p={ps2:.4f})")

print('\n--- Per-subject optimal params ---')
print(f"{'Subject':<12}{'AR':>4}{'Ctx':>5}{'Low':>5}{'High':>6}{'Ord':>4}{'Pad':>5}{'Peak':>6}")
for r in results:
    p = r['best_params']; pk = r['theta_peak_hz']
    print(f"{r['name']:<12}{p['ar_order']:>4}{p['context_s']:>5.2f}{p['filt_low']:>5.2f}"
          f"{p['filt_high']:>6.2f}{p['filt_order']:>4}{p['hilbert_pad']:>5}{(f'{pk:.2f}' if pk else 'n/a'):>6}")

out = {'model': 'BOB',
       'config': {'fs': FS, 'pre_samp': PRE_SAMP, 'eval_times_s': EVAL_TIMES_S, 'horizon_ms': HORIZON_MS,
                  'gt_band': [GT_LOW, GT_HIGH], 'gt_fir_numtaps': GT_FIR_NUMTAPS, 'n_bayes_calls': N_BAYES_CALLS,
                  'split': 'chronological cal/test from npz split array', 'conditions': ['default', 'opt'],
                  'itpc_basis': 'GT-phase ITPC, test trials, per eval point', 'errors_unit': 'radians'},
       'group_stats': group_stats,
       'subjects': results}
outpath = os.path.join(OUTDIR, 'bob_theta_results.json')
with open(outpath, 'w') as f:
    json.dump(out, f, indent=2)
print(f"\nSaved -> {outpath}")
print("Each subject holds default + opt metrics, per-timepoint errors (rad), SNR, ITPC, and BOB params.")
print("Comparators: emit the same schema with their own 'model' tag, then load all and join on 'name'.")

BOB THETA — DEFAULT vs OPTIMIZED (Opt_o)
  Default   MACE 54.4°±4.4   Acc 69.8%
  Optimized MACE 48.4°±3.4   PLV 0.561   CSD 61.7°   Acc 73.1%
  Optimization gain: +6.0° MACE  (20/20 improved)
  (n=20 unique participants -- independent samples; group/inferential stats valid)

--- Paired default vs optimized (per-subject MACE, test set) ---
  mean dMACE +6.04°   median +5.18°   (20/20 improved)
  Wilcoxon signed-rank: W=0.0, p=1.907e-06  (rank-biserial r=+1.000)
  Paired t-test:        t=+7.49, p=4.437e-07  (Cohen's dz=+1.674)
  Accuracy (opt-def):   median +2.88 pts, Wilcoxon p=1.907e-06

--- Per-timepoint (default vs optimized) ---
   ms  defMACE  optMACE  optPLV  optCSD  optAcc    ITPC      N
  300    53.2°    47.2°   0.581   59.8°   73.8%   0.121     50
  400    57.0°    49.8°   0.545   63.2°   72.3%   0.130     50
  500    54.9°    49.4°   0.552   62.6°   72.6%   0.157     50
  600    53.9°    48.0°   0.573   60.5°   73.3%   0.157     50
  700    53.2°    47.5°   0.578   60.1°   73